# 📄 İÜC_HAYEF_AI4EDU_2026

## Eğitim Süreçlerinde LLM API Entegrasyonu


📌 **Ne yapacağız:**
1. Sistemi kurup API bağlantısını yapacağız
2. Asistana **ROL**, **GÖREV** ve **KURALLAR** tanımlayacağız
3. PDF, Word veya Excel **belgemizi yükleyeceğiz**
4. Makaleyi **Yapılandırılmış Tabloya Çevirme Belge'ye** çevireceğiz.
5. **Ölçme Değerlendirme, Soru Kalite Analizi** yapacağız.

⚠️ **Programlama bilgisi gerekmez!** `👇` işareti olan yerleri değiştirmeniz yeterli.

---

---
## 📦 Adım 1: Kurulum

Gerekli araçları yükler. **Hiçbir şeyi değiştirmeyin**, sadece çalıştırın.

In [1]:
# ============================================================
# ADIM 1: Gerekli kütüphaneleri yükle
# Bu hücreyi olduğu gibi çalıştırın — hiçbir şeyi değiştirmeyin
# ============================================================

!pip install openai openpyxl pandas pdfplumber python-docx -q

from openai import OpenAI
import pandas as pd
import pdfplumber
import os
from datetime import datetime

print("✅ Kurulum tamamlandı!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 51.8 MB/s eta 0:00:00
✅ Kurulum tamamlandı!


gfgh

---
## 🔑 Adım 2: API Anahtarı

Tüm uygulamalarda gerekli olan GITHUB_API_KEY_OLUSTURMA_ADIMLARI için **[buradaki](https://drive.google.com/file/d/1xGwkhzxH9adGmItFWPQWUmCbVL7fNKpb/view?usp=sharing)** yönergeleri takip ediniz ve GITHUB_TOKEN değerini kaydediniz.

⚠️ **Anahtarınızı kimseyle paylaşmayın!**

In [2]:
# ============================================================
# ADIM 2: API anahtarınızı girin
# 👇 Sadece tırnak içindeki token kısmı değiştirin
# ============================================================

from openai import OpenAI

token = "" #os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "deepseek/DeepSeek-R1"


client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

print("✅ API bağlantısı hazır!")

✅ API bağlantısı hazır!


---
## 🎭 Adım 3: Asistanın Kimliğini Tanımlayın

Burada 4 şey tanımlıyoruz:

| Parametre | Ne İşe Yarar? | Değiştirmeli misiniz? |
|-----------|--------------|----------------------|
| **`ASISTAN_ADI`** | Sohbette kendini nasıl tanıtacak | ✅ İsterseniz değiştirin |
| **`ROL`** | Kim olduğu, uzmanlık alanı | ✅ İhtiyacınıza göre değiştirin |
| **`GÖREV`** | Belgeyle ne yapacağı | ✅ İhtiyacınıza göre değiştirin |
| **`KURALLAR`** | Nasıl davranacağı, kısıtlamalar | ✅ Ekleyip çıkarabilirsiniz |
| **`MODEL`** | Hangi yapay zekâ modeli | ✅ gpt-5-mini veya gpt-4o-mini |

### 💡 İpucu
Kurallar ne kadar net olursa, asistan o kadar tutarlı davranır.

In [3]:
# ============================================================
# ADIM 3: Asistanın kimliğini tanımlayın
#
# 👇 Aşağıdaki alanları ihtiyacınıza göre değiştirin
#    Her alanın ne işe yaradığı yanında açıklanmıştır
# ============================================================

# ---------- ASISTAN ADI ----------
# Sohbet başlarken ve bitirirken kullanılacak isim

ASISTAN_ADI = "İstanbul Üniversitesi-Cerrahpaşa Asistanı"   # 👈 İsmi değiştirebilirsiniz

# ---------- ROL ----------
# Asistanın kim olduğunu tanımlayın.
# Ne kadar spesifik olursanız, o kadar iyi sonuç alırsınız.

ROL = """
Sen bir öğretim görevlisinin doküman analiz asistanısın.
Adın: {asistan_adi}.
Görevin, sana verilen belgeleri analiz etmek, özetlemek,
karşılaştırmak ve belgeler hakkındaki soruları cevaplamaktır.
""".format(asistan_adi=ASISTAN_ADI)

# ---------- GÖREV ----------
# Belgeyle ne yapmasını istiyorsunuz?

GOREV = """
Kullanıcının yüklediği belgeyi dikkatlice oku ve anla.
Kullanıcı bu belge hakkında sorular soracak.
Her soruya belgede bulunan bilgilere dayanarak cevap ver.
Eğer belgede olmayan bir bilgi sorulursa, açıkça
"Bu bilgi belgede bulunmamaktadır" de.
"""

# ---------- KURALLAR ----------
# Asistanın uyması gereken kurallar.
# İhtiyacınıza göre madde ekleyip çıkarabilirsiniz.

KURALLAR = """
Kuralların:
1. Her zaman Türkçe cevap ver
2. Cevaplarını belgede bulunan bilgilere dayandır
3. Belgede olmayan bilgiyi uydurma (halüsinasyon yapma)
4. Sayısal veriler sorulduğunda tablo formatında cevap ver
5. Özetleme istendiğinde madde madde yaz
6. Resmi ve profesyonel bir dil kullan
7. Emin olmadığın bilgiyi "kesin" gibi sunma, belirsizliği belirt
8. Her cevapta ilgili bölümü veya sayfa numarasını referans göster
9. en sonda parantez içinde magazinel bakımdan bir paragraf yaz

"""

# ---------- MODEL ----------
# openai/gpt-5     → En güçlü, daha pahalı (uzun belgeler için önerilir)
# openai/o4-mini  → Hızlı ve ucuz, çoğu iş için yeterli

MODEL = "openai/gpt-4o-mini"   # 👈 Değiştirebilirsiniz

# ---------- SİSTEM MESAJI ----------
# Yukarıdakilerin birleşimi — değiştirmenize gerek yok

SISTEM_MESAJI = f"""{ROL}

{GOREV}

{KURALLAR}"""

print("✅ Asistan kimliği tanımlandı!")
print(f"   Ad: {ASISTAN_ADI}")
print(f"   Model: {MODEL}")
print(f"   Kural sayısı: {KURALLAR.strip().count(chr(10))}")

✅ Asistan kimliği tanımlandı!
   Ad: İstanbul Üniversitesi-Cerrahpaşa Asistanı
   Model: openai/gpt-4o-mini
   Kural sayısı: 9


---
## 📂 Adım 4: Belgenizi Yükleyin

Bu hücre dosya seçim penceresini açar. Belgenizi seçin, otomatik olarak okunacaktır.

### Desteklenen Formatlar
| Format | Uzantı | Açıklama |
|--------|--------|----------|
| PDF | `.pdf` | Faaliyet raporları, yönetmelikler, yönergeler |
| Word | `.docx` | Yazışmalar, tutanaklar, raporlar |
| Excel | `.xlsx` | Tablolar, listeler, anket sonuçları |
| CSV | `.csv` | Veri dosyaları |
| Metin | `.txt` | Düz metin dosyaları |

⚠️ **Uyarı:** Gizli veya kişisel veri içeren belgelerde dikkatli olun. Veriler OpenAI sunucularına gönderilir.

In [4]:
# ============================================================
# ADIM 4: Belgenizi yükleyin
#
# Bu hücreyi çalıştırın → Dosya seçim penceresi açılır
# Belgenizi seçin → Otomatik olarak okunur
# ============================================================

def belge_oku(dosya_adi, icerik_bytes):
    """
    Farklı formatlardaki belgeleri okur ve metin olarak döndürür.
    """
    uzanti = dosya_adi.lower().split('.')[-1]

    if uzanti == 'pdf':
        # PDF dosyasını kaydet ve pdfplumber ile oku
        with open(dosya_adi, 'wb') as f:
            f.write(icerik_bytes)
        metin = ""
        with pdfplumber.open(dosya_adi) as pdf:
            for i, sayfa in enumerate(pdf.pages):
                sayfa_metni = sayfa.extract_text()
                if sayfa_metni:
                    metin += f"\n--- Sayfa {i+1} ---\n{sayfa_metni}"
                # Tabloları da çıkar
                tablolar = sayfa.extract_tables()
                for j, tablo in enumerate(tablolar):
                    metin += f"\n[Tablo {j+1}, Sayfa {i+1}]\n"
                    for satir in tablo:
                        metin += " | ".join([str(h) if h else "" for h in satir]) + "\n"
        return metin

    elif uzanti == 'docx':
        from docx import Document
        with open(dosya_adi, 'wb') as f:
            f.write(icerik_bytes)
        doc = Document(dosya_adi)
        return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

    elif uzanti in ('xlsx', 'xls'):
        with open(dosya_adi, 'wb') as f:
            f.write(icerik_bytes)
        tum_metin = ""
        xls = pd.ExcelFile(dosya_adi)
        for sayfa_adi in xls.sheet_names:
            df = pd.read_excel(dosya_adi, sheet_name=sayfa_adi)
            tum_metin += f"\n--- Sayfa: {sayfa_adi} ---\n{df.to_string()}\n"
        return tum_metin

    elif uzanti == 'csv':
        with open(dosya_adi, 'wb') as f:
            f.write(icerik_bytes)
        df = pd.read_csv(dosya_adi)
        return df.to_string()

    else:
        # txt ve diğer metin dosyaları
        return icerik_bytes.decode('utf-8', errors='replace')


# --- DOSYA YÜKLEME ---
belge_metni = None
belge_adi = None

try:
    from google.colab import files
    print("📂 Dosya seçim penceresi açılıyor...")
    print("   Desteklenen: PDF, DOCX, XLSX, CSV, TXT\n")
    yuklenen = files.upload()

    for dosya_adi, icerik in yuklenen.items():
        belge_adi = dosya_adi
        print(f"📄 Yüklenen: {dosya_adi} ({len(icerik):,} byte)")
        print("⏳ Belge okunuyor...")

        belge_metni = belge_oku(dosya_adi, icerik)

        print(f"✅ Belge başarıyla okundu!")
        print(f"   Toplam karakter: {len(belge_metni):,}")
        print(f"   Tahmini kelime: ~{len(belge_metni.split()):,}")
        print(f"\n{'─' * 60}")
        print(f"İlk 1000 karakter:\n")
        print(belge_metni[:1000])
        print(f"\n{'─' * 60}")
        print("\n✅ Belge hazır! Şimdi Adım 5'i çalıştırarak soru sormaya başlayabilirsiniz.")
        break  # Sadece ilk dosyayı al

except ImportError:
    print("⚠️ Dosya yükleme sadece Google Colab'da çalışır.")
    print("   Alternatif: Belge metnini aşağıdaki hücreye manuel yapıştırabilirsiniz.")

📂 Dosya seçim penceresi açılıyor...
   Desteklenen: PDF, DOCX, XLSX, CSV, TXT



Saving Fine-tuning_Wav2Vec2_for_Classification_of_Turkish_Broadcast_News_and_Advertisement_Jingles.pdf to Fine-tuning_Wav2Vec2_for_Classification_of_Turkish_Broadcast_News_and_Advertisement_Jingles.pdf
📄 Yüklenen: Fine-tuning_Wav2Vec2_for_Classification_of_Turkish_Broadcast_News_and_Advertisement_Jingles.pdf (283,681 byte)
⏳ Belge okunuyor...
✅ Belge başarıyla okundu!
   Toplam karakter: 31,669
   Tahmini kelime: ~3,644

────────────────────────────────────────────────────────────
İlk 1000 karakter:


--- Sayfa 1 ---
Fine-tuning Wav2Vec2 for Classification of Turkish
Broadcast News and Advertisement Jingles
1st Ferhat Demirkıran * 2nd Onur Öner 3rd Yavuz Kömeçog˘lu
Management Information Systems Kodiks Bilis¸im A.S¸ Kodiks Bilis¸im A.S¸
Kadir Has University R&D Center R&D Center
˙Istanbul, Turkey ˙Istanbul, Turkey ˙Istanbul, Turkey
ferhat.demirkiran@khas.edu.tr onur.oner@kodiks.com yavuz.komecoglu@kodiks.com
4th Ramazan Güven 5th Bas¸ak Buluz Kömeçog˘lu
Kodiks Bilis¸im A.S¸ Computer En

### 📝 Alternatif: Belgeyi Manuel Girin (Opsiyonel)

Eğer dosya yükleme çalışmazsa veya kısa bir metin analiz ettirmek istiyorsanız,
belge içeriğini aşağıya yapıştırabilirsiniz.

In [5]:
# ============================================================
# ALTERNATİF: Belge metnini buraya yapıştırın (opsiyonel)
#
# Eğer dosya yükleme çalışmazsa bu hücreyi kullanın.
# 👇 Üç tırnak arasına belge metnini yapıştırın
# ============================================================

# Bu hücreyi SADECE dosya yükleme çalışmazsa kullanın.
# Kullanmak istemiyorsanız bu hücreyi atlayın.

belge_metni = """
I. INTRODUCTION
In today’s world, television and radio broadcasts are followed by a wide audience. These broadcasts consist of various
types of programs, advertisements, news, series, and movies.
Extracting Electronic Program Guide (EPG) information enables the identification of different types of programs, advertisements, and other content in a television or radio broadcast,
including their genre, title, start, and end times. Obtaining
this information allows for faster, automatic, and reliable
cataloging and searching of television and radio broadcasts.
This, in turn, facilitates viewers in finding the programs they
are looking for more easily, while also assisting broadcasting companies and media monitoring firms in archiving and
conducting retrospective searches more efficiently [1], [2].
Manually extracting EPG information can be a costly
process in terms of workforce and resources, considering
*Corresponding author: ferhat.demirkiran@khas.edu.tr
factors such as the type, duration, and content of broadcasts.
Moreover, since it is prone to human errors, the accuracy rate
may be low. Automatic data extraction methods are designed
to make this process faster, more accurate, and more reliable.
These methods utilize technologies such as image processing,
audio processing, natural language processing to automatically
analyze the content and characteristics of broadcasts. This
allows for quick and accurate retrieval of data such as program
genres, program titles, start and end times [3].
The detection of news and advertisement music is crucial for accurately extracting EPG information. In television

"""
belge_adi = "manuel_belge.txt"
print(f"✅ Manuel belge yüklendi! ({len(belge_metni):,} karakter)")

✅ Manuel belge yüklendi! (1,627 karakter)


---
## 💬 Adım 5: Soru-Cevap Sistemini Başlatın

Bu hücreyi çalıştırınca interaktif sohbet başlar.

### Nasıl Kullanılır?
- Sorularınızı yazıp **Enter**'a basın
- Asistan belgeden bilgi bulup cevap verecek
- Birden fazla soru sorabilirsiniz — asistan önceki soruları hatırlar
- Çıkmak için **`exit`** veya **`çıkış`** yazın

### 💡 Örnek Sorular
- *"Bu belgenin ana konusu nedir?"*
- *"Personel sayıları hakkında bilgi ver"*
- *"Tablodaki verileri özetle"*
- *"Bu belgeden yöneticiye sunulacak 3 maddelik özet çıkar"*
- *"Belgede dikkat çekici veya sorunlu noktalar var mı?"*

In [6]:
# ============================================================
# ADIM 5: Soru-Cevap sistemini başlat
#
# Bu hücreyi çalıştırın → Sohbet başlar
# Sorularınızı yazın → Enter'a basın
# Çıkmak için: exit veya çıkış yazın
# ============================================================

# Belge yüklendi mi kontrol et
if belge_metni is None or len(str(belge_metni).strip()) == 0:
    print("❌ Henüz bir belge yüklenmedi!")
    print("   Lütfen önce Adım 4'ü çalıştırıp belgenizi yükleyin.")
else:
    # --- Sohbet geçmişi ---
    # İlk mesaj: sistem mesajı + belge içeriği
    sohbet_gecmisi = [
        {
            "role": "system",
            "content": f"""{SISTEM_MESAJI}

Aşağıda analiz edeceğin belge var:

--- BELGE BAŞLANGIÇ ---
Belge Adı: {belge_adi}

{belge_metni[:50000]}
--- BELGE BİTİŞ ---

Kullanıcı bu belge hakkında sorular soracak. Her soruyu belgede bulunan
bilgilere dayanarak cevapla."""
        }
    ]

    # --- Soru-cevap kaydı (Excel için) ---
    soru_cevap_kaydi = []

    # --- Token sayacı ---
    toplam_token = 0

    # --- Karşılama mesajı ---
    print("╔" + "═" * 58 + "╗")
    print("║" + f" 📄 {ASISTAN_ADI} ".center(58) + "║")
    print("╠" + "═" * 58 + "╣")
    print("║" + " İstanbul Üniversitesi-Cerrahpaşa Yapay Zeka Ajanı: Doküman Analiz ve Soru-Cevap Sistemi".ljust(58) + "║")
    print("╚" + "═" * 58 + "╝")
    print()

    # Asistanın kendini tanıtması
    tanitim_mesaji = f"""Merhaba! Ben {ASISTAN_ADI}. 🤖

📄 Yüklenen belge: {belge_adi}
📊 Belge boyutu: {len(belge_metni):,} karakter (~{len(belge_metni.split()):,} kelime)

Bu belge hakkında istediğiniz soruyu sorabilirsiniz.
Özetleme, analiz, karşılaştırma, hesaplama — her konuda yardımcı olurum.

💡 Örnek sorular:
   • "Bu belgenin ana konusu nedir?"
   • "Personel bilgilerini özetle"
   • "Belgeden yöneticiye sunulacak 3 madde çıkar"

Çıkmak için 'exit' veya 'çıkış' yazabilirsiniz.
"""
    print(tanitim_mesaji)
    print("─" * 60)

    # --- Soru-Cevap Döngüsü ---
    soru_no = 0

    while True:
        # Kullanıcıdan soru al
        print()
        soru = input("🧑 Siz: ").strip()

        # Boş soru kontrolü
        if not soru:
            print("⚠️  Lütfen bir soru yazın.")
            continue

        # Çıkış kontrolü
        if soru.lower() in ['exit', 'çıkış', 'quit', 'kapat', 'bitir', 'q']:
            print()
            print("─" * 60)
            veda = f"""👋 Görüşmek üzere!

📊 Sohbet Özeti:
   • Toplam soru: {soru_no}
   • Toplam token: {toplam_token:,}
   • Tahmini maliyet: ~${toplam_token * 0.00000015:.4f}
   • Belge: {belge_adi}

İyi günler dilerim! Tekrar sormak istediğiniz bir şey olursa
bu hücreyi yeniden çalıştırmanız yeterli.

— {ASISTAN_ADI} 🤖"""
            print(veda)
            print("─" * 60)
            break

        soru_no += 1

        # Kullanıcı mesajını geçmişe ekle
        sohbet_gecmisi.append({"role": "user", "content": soru})

        # API'ye gönder
        try:
            print(f"\n⏳ Düşünüyorum...\n")

            cevap = client.chat.completions.create(
                model=MODEL,
                messages=sohbet_gecmisi,
                temperature=0.3  # Düşük = daha tutarlı (belge analizi için ideal)
            )

            asistan_cevabi = cevap.choices[0].message.content
            kullanim = cevap.usage
            toplam_token += kullanim.total_tokens

            # Asistan cevabını geçmişe ekle (hafıza)
            sohbet_gecmisi.append({"role": "assistant", "content": asistan_cevabi})

            # Cevabı göster
            print(f"🤖 {ASISTAN_ADI}:")
            print()
            print(asistan_cevabi)
            print()
            print(f"   📊 Token: {kullanim.total_tokens:,} | Toplam: {toplam_token:,} | ~${toplam_token * 0.00000015:.4f}")
            print("─" * 60)

            # Kayıt tut
            soru_cevap_kaydi.append({
                "soru_no": soru_no,
                "zaman": datetime.now().strftime("%H:%M:%S"),
                "soru": soru,
                "cevap": asistan_cevabi,
                "token": kullanim.total_tokens
            })

        except Exception as e:
            print(f"❌ Hata: {e}")
            print("   API anahtarınızı ve internet bağlantınızı kontrol edin.")
            # Hatalı mesajı geçmişten çıkar
            sohbet_gecmisi.pop()

    print("\n✅ Sohbet tamamlandı! Kaydetmek için Adım 6'yı çalıştırın.")

╔══════════════════════════════════════════════════════════╗
║       📄 İstanbul Üniversitesi-Cerrahpaşa Asistanı        ║
╠══════════════════════════════════════════════════════════╣
║ İstanbul Üniversitesi-Cerrahpaşa Yapay Zeka Ajanı: Doküman Analiz ve Soru-Cevap Sistemi║
╚══════════════════════════════════════════════════════════╝

Merhaba! Ben İstanbul Üniversitesi-Cerrahpaşa Asistanı. 🤖

📄 Yüklenen belge: manuel_belge.txt
📊 Belge boyutu: 1,627 karakter (~228 kelime)

Bu belge hakkında istediğiniz soruyu sorabilirsiniz.
Özetleme, analiz, karşılaştırma, hesaplama — her konuda yardımcı olurum.

💡 Örnek sorular:
   • "Bu belgenin ana konusu nedir?"
   • "Personel bilgilerini özetle"
   • "Belgeden yöneticiye sunulacak 3 madde çıkar"

Çıkmak için 'exit' veya 'çıkış' yazabilirsiniz.

────────────────────────────────────────────────────────────

🧑 Siz: makalenin özetini verir misin

⏳ Düşünüyorum...

🤖 İstanbul Üniversitesi-Cerrahpaşa Asistanı:

Elbette, işte belgenin özeti:

- Günümüzde 

KeyboardInterrupt: Interrupted by user

---
## 💾 Adım 6: Makaleden Yapılandırılmış Veri Çıkartma


In [9]:
response = client.chat.completions.create(
    messages=[
          {
              "role": "system",
              "content": "Amaç, Örneklem, Yöntem,Bulgular'ı analiz et ve JSON Formatında Türkçe dilinde ver.",
          },
          {
              "role": "user",
              "content": belge_metni,
          }
      ],
      model=MODEL,
      temperature=0.3
  )

from IPython.display import Markdown
display(Markdown(response.choices[0].message.content))

```json
{
  "Amaç": "Elektronik Program Rehberi (EPG) bilgilerini otomatik olarak çıkarmak ve bu sayede televizyon ve radyo yayınlarının daha hızlı, doğru ve güvenilir bir şekilde kataloglanmasını sağlamak.",
  "Örneklem": "Televizyon ve radyo yayınları, programlar, reklamlar, haberler, diziler ve filmler.",
  "Yöntem": "Görüntü işleme, ses işleme ve doğal dil işleme teknolojileri kullanılarak yayınların içeriği ve özellikleri otomatik olarak analiz edilmektedir.",
  "Bulgular": {
    "Faydalar": [
      "İzleyicilerin aradıkları programları daha kolay bulmalarını sağlamak.",
      "Yayıncı şirketler ve medya izleme firmalarının arşivleme ve geriye dönük arama süreçlerini daha verimli hale getirmek.",
      "Manuel EPG bilgisi çıkarmanın iş gücü ve kaynak açısından maliyetli olduğunu belirtmek.",
      "Otomatik veri çıkarım yöntemlerinin süreci daha hızlı, daha doğru ve daha güvenilir hale getirdiğini vurgulamak."
    ],
    "Sorunlar": [
      "Manuel çıkarımın insan hatalarına açık olması ve doğruluk oranının düşük olabilmesi."
    ]
  }
}
```

---
## 💾 Adım 7: Sınav Sorusu Kalite Denetçisi


In [10]:
belge_metni = """
Türkiye’nin başkenti hangisidir?

A) Ankara
B) Kalem
C) Masa
D) Defter

Doğru Cevap: A

Aşağıdakilerden hangisi memelidir?

A) Kedi
B) Yunus
C) Köpek
D) Hepsi

Doğru Cevap: D

En iyi ulaşım aracı hangisidir?

A) Araba
B) Bisiklet
C) Metro
D) Uçak

Doğru Cevap: Belirsiz

Aşağıdakilerden hangisi bireylerin günlük yaşam içerisindeki sosyal iletişim süreçlerini daha etkili ve verimli bir biçimde gerçekleştirebilmesine katkı sağlayabilecek davranış örneklerinden biri olarak değerlendirilebilir?

A) Saygılı konuşmak
B) Bağırmak
C) Hakaret etmek
D) İnsanları küçümsemek

Doğru Cevap: A

Aşağıdakilerden hangisi bir gezegendir?

A) Mars
B) Gezegen olmayan Mars
C) Kırmızı Mars gezegeni
D) Uzay topu

Doğru Cevap: A

2 + 2 kaç eder?

A) 4
B) 400
C) Elma
D) Masa

Doğru Cevap: A

Aşağıdakilerden hangisi değildir olmayan bir canlı değildir?

A) Kedi
B) Taş
C) İnsan
D) Köpek

Doğru Cevap: Belirsiz

Hangisi asal sayıdır?

A) 2
B) 3
C) 4
D) 6

Doğru Cevap: A ve B

Aşağıdakilerden hangisi enerji kaynağıdır?

A) Kömür
B) Elektrik üretim süreçlerinde yaygın biçimde kullanılan fosil yakıt türlerinden biri olan kömür
C) Masa
D) Kalem

Doğru Cevap: A

Hangisi bir meyvedir?

A) Elma
B) ARMUT
C) muz
D) Bilgisayar

Doğru Cevap: Belirsiz (A, B ve C)
"""




response = client.chat.completions.create(
    messages=[
          {
              "role": "system",
              "content": """
                  Aşağıdaki çoktan seçmeli soruları ölçme-değerlendirme uzmanı gibi analiz et.

                  Her soru için şu kriterleri değerlendir:

                  1. Çeldiriciler mantıklı mı?
                  2. Tek doğru cevap var mı?
                  3. Dil sade ve anlaşılır mı?
                  4. Bloom taksonomisine göre bilişsel düzeyi nedir?
                  5. Sorunun genel kalite puanı (1-10)

                  Sonuçları tablo halinde ver.

                  Daha sonra:
                  - Her soruyu daha kaliteli olacak şekilde yeniden yaz.
                  - Orijinal ve yeni versiyonu karşılaştır.
                  - Yapılan iyileştirmeleri açıkla.
              """,
          },
          {
              "role": "user",
              "content": belge_metni,
          }
      ],
      model=MODEL,
      temperature=0.3
  )

from IPython.display import Markdown
display(Markdown(response.choices[0].message.content))

### Analiz Tablosu

| Soru No | Çeldiriciler Mantıklı mı? | Tek Doğru Cevap Var mı? | Dil Sade ve Anlaşılır mı? | Bloom Taksonomisine Göre Bilişsel Düzeyi | Genel Kalite Puanı (1-10) |
|---------|---------------------------|-------------------------|---------------------------|------------------------------------------|---------------------------|
| 1       | Evet                      | Evet                    | Evet                      | Bilgi (Hatırlama)                       | 8                         |
| 2       | Evet                      | Evet                    | Evet                      | Bilgi (Hatırlama)                       | 8                         |
| 3       | Evet                      | Hayır                   | Evet                      | Bilgi (Hatırlama)                       | 5                         |
| 4       | Evet                      | Evet                    | Evet                      | Uygulama (Uygulama)                    | 8                         |
| 5       | Evet                      | Evet                    | Evet                      | Bilgi (Hatırlama)                       | 8                         |
| 6       | Evet                      | Evet                    | Evet                      | Bilgi (Hatırlama)                       | 8                         |
| 7       | Hayır                    | Hayır                   | Evet                      | Bilgi (Hatırlama)                       | 4                         |
| 8       | Evet                      | Hayır                   | Evet                      | Bilgi (Hatırlama)                       | 5                         |
| 9       | Evet                      | Evet                    | Evet                      | Bilgi (Hatırlama)                       | 8                         |
| 10      | Evet                      | Hayır                   | Evet                      | Bilgi (Hatırlama)                       | 5                         |

### Yeniden Yazım ve Karşılaştırma

#### Soru 1
**Orijinal:** Türkiye’nin başkenti hangisidir?  
**Yeni:** Türkiye'nin başkenti nedir?  
**İyileştirme:** "Hangisi" yerine "nedir" kullanarak daha doğrudan bir ifade sağlandı.

#### Soru 2
**Orijinal:** Aşağıdakilerden hangisi memelidir?  
**Yeni:** Aşağıdaki hayvanlardan hangisi memeli bir türdür?  
**İyileştirme:** "Memelidir" ifadesi daha açıklayıcı hale getirildi.

#### Soru 3
**Orijinal:** En iyi ulaşım aracı hangisidir?  
**Yeni:** En etkili ulaşım aracı hangisidir?  
**İyileştirme:** "En iyi" ifadesi "en etkili" olarak değiştirildi, böylece daha net bir kıyaslama sağlandı.

#### Soru 4
**Orijinal:** Aşağıdakilerden hangisi bireylerin günlük yaşam içerisindeki sosyal iletişim süreçlerini daha etkili ve verimli bir biçimde gerçekleştirebilmesine katkı sağlayabilecek davranış örneklerinden biri olarak değerlendirilebilir?  
**Yeni:** Aşağıdaki davranışlardan hangisi sosyal iletişimi daha etkili hale getirir?  
**İyileştirme:** Sorunun uzunluğu kısaltıldı ve daha net bir ifade kullanıldı.

#### Soru 5
**Orijinal:** Aşağıdakilerden hangisi bir gezegendir?  
**Yeni:** Aşağıdaki seçeneklerden hangisi bir gezegen olarak kabul edilir?  
**İyileştirme:** "Hangisi bir gezegendir" ifadesi daha açıklayıcı hale getirildi.

#### Soru 6
**Orijinal:** 2 + 2 kaç eder?  
**Yeni:** 2 ile 2'nin toplamı nedir?  
**İyileştirme:** Daha açık bir matematiksel ifade kullanıldı.

#### Soru 7
**Orijinal:** Aşağıdakilerden hangisi değildir olmayan bir canlı değildir?  
**Yeni:** Aşağıdakilerden hangisi canlı değildir?  
**İyileştirme:** Sorunun mantığı sadeleştirildi ve daha anlaşılır hale getirildi.

#### Soru 8
**Orijinal:** Hangisi asal sayıdır?  
**Yeni:** Aşağıdaki sayılardan hangisi asal bir sayıdır?  
**İyileştirme:** Daha net bir ifade kullanıldı.

#### Soru 9
**Orijinal:** Aşağıdakilerden hangisi enerji kaynağıdır?  
**Yeni:** Aşağıdaki seçeneklerden hangisi bir enerji kaynağıdır?  
**İyileştirme:** Daha açık bir ifade kullanıldı.

#### Soru 10
**Orijinal:** Hangisi bir meyvedir?  
**Yeni:** Aşağıdaki seçeneklerden hangisi bir meyve olarak kabul edilir?  
**İyileştirme:** Daha net bir ifade kullanıldı.

### Genel İyileştirmeler
- Soruların dil yapısı sadeleştirildi ve daha anlaşılır hale getirildi.
- Soruların mantığı ve yapılandırılması gözden geçirilerek netlik artırıldı.
- Bazı soruların uzunluğu kısaltılarak, gereksiz karmaşıklıklar ortadan kaldırıldı.
- Soruların bilişsel düzeyleri daha iyi ifade edildi.

---
## 📋 Hızlı Referans: Neyi Nerede Değiştirmeliyim?

| Ne Yapmak İstiyorsunuz? | Hangi Adım? | Neyi Değiştirin? |
|------------------------|-------------|------------------|
| API anahtarımı girmek | Adım 2 | `API_ANAHTARI = "sk-..."` |
| Asistanın adını değiştirmek | Adım 3 | `ASISTAN_ADI = "..."` |
| Asistanın rolünü değiştirmek | Adım 3 | `ROL = """..."""` içindeki metin |
| Görevini değiştirmek | Adım 3 | `GOREV = """..."""` içindeki metin |
| Kural eklemek/çıkarmak | Adım 3 | `KURALLAR = """..."""` içindeki maddeler |
| Farklı model kullanmak | Adım 3 | `MODEL = "gpt-4o"` veya `"gpt-4o-mini"` |
| Belge yüklemek | Adım 4 | Hücreyi çalıştırıp dosya seçin |
| Soru sormaya başlamak | Adım 5 | Hücreyi çalıştırıp sorularınızı yazın |
| Çıkmak | Adım 5 | `exit` veya `çıkış` yazın |
| Sonuçları kaydetmek | Adım 6 | Hücreyi çalıştırın |

### 💡 İpuçları
- **Soru ne kadar spesifik olursa, cevap o kadar iyi olur**
- Asistan **önceki soruları hatırlar** — "Bunu tablo halinde göster" gibi takip soruları sorabilirsiniz
- **"Belgede bu bilgi var mı?"** diye sorarak halüsinasyon kontrolü yapabilirsiniz
- Uzun belgelerde ilk sorunuz **"Bu belgenin ana konularını 5 maddede özetle"** olsun
- Her oturumdan sonra **Adım 6 ile Excel'e kaydedin** — sohbet geçmişiniz korunur

### ⚠️ Güvenlik Hatırlatması
- Kişisel veri (TC, telefon, öğrenci bilgisi) içeren belgelerde dikkatli olun
- Gizli kurumsal belgeleri ücretsiz araçlara yüklemeyin
- API ile gönderilen veriler OpenAI sunucularından geçer

---
**Yavuz Kömeçoğlu - ThinkVoice Kurucu Mühendisi & Kodiks Ar-Ge Mühendisi**

**(yavuz.komecoglu@kodiks.com)**  
**Mayıs 2026**